# Solar Energy Production Forecasting: Weather Impact Analysis
This notebook examines the influence of meteorological variables on solar panel energy output and develops a Random Forest regression model for daily energy production forecasting.

## 1. Setup & Data Upload

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

from google.colab import files
uploaded = files.upload()

## 2. Load & Prepare Data

In [ ]:
df = pd.read_csv('daily_aggregated.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df[df['energy'] > 0].copy()

df_daily = df.groupby('timestamp').agg({
    'energy': 'sum',
    'irradiance': 'first',
    'sunshine_hours': 'first',
    'temperature_2m_max': 'first',
    'ambient_temp': 'first',
    'humidity': 'first',
    'precipitation_sum': 'first',
    'cloud_cover_mean': 'first',
    'wind_speed': 'first',
    'clearness_index': 'first',
    'temp_range': 'first'
}).reset_index()

df_daily['day_of_year'] = df_daily['timestamp'].dt.dayofyear
df_daily['month'] = df_daily['timestamp'].dt.month

print(f"Records: {len(df_daily)}")
print(f"Date range: {df_daily['timestamp'].min().date()} to {df_daily['timestamp'].max().date()}")
print(f"Columns: {list(df_daily.columns)}")
df_daily.head()

## 3. Weather Category Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

if 'cloud_cover_mean' in df_daily.columns:
    df_daily['cloud_cat'] = pd.cut(df_daily['cloud_cover_mean'],
                                    bins=[0, 40, 70, 100],
                                    labels=['Clear', 'Partly Cloudy', 'Cloudy'])
    ax = axes[0]
    cloud_stats = df_daily.groupby('cloud_cat')['energy'].agg(['mean', 'std']).reset_index()
    cloud_stats = cloud_stats.dropna()
    if len(cloud_stats) > 0:
        ax.bar(cloud_stats['cloud_cat'].astype(str), cloud_stats['mean'],
               yerr=cloud_stats['std'], capsize=5, color=['#99ccff', '#cccccc', '#666666'])
        ax.set_xlabel('Cloud Cover')
        ax.set_ylabel('Mean Energy (kWh)')
        ax.set_title('Energy by Cloud Cover')
        print("Energy by Cloud Cover:")
        print(cloud_stats)

if 'precipitation_sum' in df_daily.columns:
    df_daily['rain_cat'] = df_daily['precipitation_sum'].apply(lambda x: 'Rainy' if x > 1 else 'Dry')
    ax = axes[1]
    rain_stats = df_daily.groupby('rain_cat')['energy'].agg(['mean', 'std']).reset_index()
    if len(rain_stats) > 0:
        ax.bar(rain_stats['rain_cat'], rain_stats['mean'],
               yerr=rain_stats['std'], capsize=5, color=['#ffcc66', '#6699cc'])
        ax.set_xlabel('Weather Condition')
        ax.set_ylabel('Mean Energy (kWh)')
        ax.set_title('Energy: Dry vs Rainy Days')
        print("\nEnergy by Precipitation:")
        print(rain_stats)

if 'ambient_temp' in df_daily.columns:
    df_daily['temp_cat'] = pd.cut(df_daily['ambient_temp'],
                                   bins=[0, 26, 28, 35],
                                   labels=['Cool', 'Moderate', 'Hot'])
    ax = axes[2]
    temp_stats = df_daily.groupby('temp_cat')['energy'].agg(['mean', 'std']).reset_index()
    temp_stats = temp_stats.dropna()
    if len(temp_stats) > 0:
        ax.bar(temp_stats['temp_cat'].astype(str), temp_stats['mean'],
               yerr=temp_stats['std'], capsize=5, color=['#99ff99', '#ffcc99', '#ff9999'])
        ax.set_xlabel('Temperature')
        ax.set_ylabel('Mean Energy (kWh)')
        ax.set_title('Energy by Temperature')
        print("\nEnergy by Temperature:")
        print(temp_stats)

plt.tight_layout()
plt.savefig('weather_categories.png', dpi=150)
plt.show()

## 4. Train Random Forest Model

In [ ]:
feature_cols = [
    'irradiance', 'sunshine_hours', 'temperature_2m_max',
    'ambient_temp', 'humidity', 'precipitation_sum',
    'cloud_cover_mean', 'wind_speed', 'clearness_index',
    'temp_range', 'day_of_year', 'month'
]

feature_cols = [c for c in feature_cols if c in df_daily.columns]
print(f"Features used: {feature_cols}")

X = df_daily[feature_cols].fillna(df_daily[feature_cols].median())
y = df_daily['energy']

split_idx = int(len(df_daily) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_test = df_daily['timestamp'].iloc[split_idx:]

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20],
    'min_samples_leaf': [1, 2]
}

rf = RandomForestRegressor(random_state=42)
grid = GridSearchCV(rf, param_grid, cv=TimeSeriesSplit(n_splits=3),
                    scoring='neg_mean_absolute_error', n_jobs=-1)
grid.fit(X_train, y_train)

model = grid.best_estimator_
print(f"Best parameters: {grid.best_params_}")

## 5. Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("Model Performance:")
print(f"  MAE:  {mae:,.2f} kWh")
print(f"  RMSE: {rmse:,.2f} kWh")
print(f"  R\u00b2:   {r2:.4f}")
print(f"  MAPE: {mape:.2f}%")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
ax.scatter(y_test, y_pred, alpha=0.6)
max_val = max(y_test.max(), max(y_pred))
ax.plot([0, max_val], [0, max_val], 'r--')
ax.set_xlabel('Actual Energy (kWh)')
ax.set_ylabel('Predicted Energy (kWh)')
ax.set_title(f'Actual vs Predicted (R\u00b2 = {r2:.3f})')
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(dates_test.values, y_test.values, 'b-o', label='Actual', markersize=4)
ax.plot(dates_test.values, y_pred, 'r-s', label='Predicted', markersize=4)
ax.set_xlabel('Date')
ax.set_ylabel('Energy (kWh)')
ax.set_title('Forecast vs Actual Over Time')
ax.legend()
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
importance = model.feature_importances_
sorted_idx = np.argsort(importance)
ax.barh(range(len(sorted_idx)), importance[sorted_idx])
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([feature_cols[i] for i in sorted_idx])
ax.set_xlabel('Importance')
ax.set_title('Feature Importance')

ax = axes[1, 1]
residuals = y_test.values - y_pred
ax.hist(residuals, bins=15, edgecolor='black', alpha=0.7)
ax.axvline(x=0, color='r', linestyle='--')
ax.set_xlabel('Residual (kWh)')
ax.set_ylabel('Frequency')
ax.set_title('Residual Distribution')

plt.tight_layout()
plt.savefig('model_performance.png', dpi=150)
plt.show()

In [ ]:
print("Feature Importance:")
for feat, imp in sorted(zip(feature_cols, importance), key=lambda x: -x[1]):
    print(f"  {feat}: {imp:.4f}")

## 6. Weather Scenario Predictions

In [ ]:
baseline = df_daily[feature_cols].median().to_dict()

scenarios = {
    'Ideal (High radiation, clear sky)': {
        'irradiance': df_daily['irradiance'].quantile(0.9) if 'irradiance' in feature_cols else None,
        'sunshine_hours': df_daily['sunshine_hours'].quantile(0.9) if 'sunshine_hours' in feature_cols else None,
        'cloud_cover_mean': df_daily['cloud_cover_mean'].quantile(0.1) if 'cloud_cover_mean' in feature_cols else None,
        'precipitation_sum': 0 if 'precipitation_sum' in feature_cols else None
    },
    'Average conditions': {},
    'Cloudy day': {
        'cloud_cover_mean': df_daily['cloud_cover_mean'].quantile(0.9) if 'cloud_cover_mean' in feature_cols else None,
        'irradiance': df_daily['irradiance'].quantile(0.3) if 'irradiance' in feature_cols else None,
        'sunshine_hours': df_daily['sunshine_hours'].quantile(0.3) if 'sunshine_hours' in feature_cols else None
    },
    'Rainy day': {
        'precipitation_sum': df_daily['precipitation_sum'].quantile(0.9) if 'precipitation_sum' in feature_cols else None,
        'cloud_cover_mean': df_daily['cloud_cover_mean'].quantile(0.9) if 'cloud_cover_mean' in feature_cols else None,
        'irradiance': df_daily['irradiance'].quantile(0.2) if 'irradiance' in feature_cols else None,
        'sunshine_hours': df_daily['sunshine_hours'].quantile(0.2) if 'sunshine_hours' in feature_cols else None
    },
    'Hot & humid': {
        'ambient_temp': df_daily['ambient_temp'].quantile(0.95) if 'ambient_temp' in feature_cols else None,
        'humidity': df_daily['humidity'].quantile(0.9) if 'humidity' in feature_cols else None
    }
}

results = []
print("Predicted Energy Production by Scenario:")
print("-" * 50)

for scenario_name, modifications in scenarios.items():
    scenario_data = baseline.copy()
    for k, v in modifications.items():
        if v is not None:
            scenario_data[k] = v
    X_scenario = pd.DataFrame([scenario_data])
    prediction = model.predict(X_scenario)[0]
    results.append((scenario_name, prediction))
    print(f"  {scenario_name:35s}: {prediction:,.0f} kWh")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
names = [r[0] for r in results]
values = [r[1] for r in results]
colors = ['#2ecc71', '#3498db', '#95a5a6', '#e74c3c', '#e67e22']

bars = ax.barh(names, values, color=colors)
ax.set_xlabel('Predicted Energy (kWh)')
ax.set_title('Predicted Energy Production Under Different Weather Scenarios')

for bar, val in zip(bars, values):
    ax.text(val + 200, bar.get_y() + bar.get_height()/2, f'{val:,.0f}', va='center')

plt.tight_layout()
plt.savefig('scenario_predictions.png', dpi=150)
plt.show()

## 7. Save Model & Download Results

In [ ]:
joblib.dump(model, 'rf_solar_model.joblib')

metrics_df = pd.DataFrame({
    'Metric': ['MAE (kWh)', 'RMSE (kWh)', 'R\u00b2', 'MAPE (%)'],
    'Value': [mae, rmse, r2, mape]
})
metrics_df.to_csv('model_metrics.csv', index=False)

print("Files saved:")
print("  - rf_solar_model.joblib")
print("  - model_metrics.csv")
print("  - weather_categories.png")
print("  - model_performance.png")
print("  - scenario_predictions.png")

In [ ]:
files.download('rf_solar_model.joblib')
files.download('model_metrics.csv')
files.download('weather_categories.png')
files.download('model_performance.png')
files.download('scenario_predictions.png')